# Phase 7B — Dashboard Data Foundation

This notebook converts the locked Phase 4–7A analytical outputs into a static dashboard data contract. It does not recalculate player performance. It adds audited national-team metadata and validates the multi-select ranking semantics used by the web dashboard.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT = ROOT / 'phase7_dashboard_data'
PUBLIC = ROOT / 'dashboard' / 'public'
OUT.mkdir(exist_ok=True)
PUBLIC.mkdir(parents=True, exist_ok=True)
MODEL_VERSION = 'dashboard_data_v1'
ROLE_ORDER = {'Batter': 0, 'Bowler': 1}
MODE_ORDER = ['IPL', 'T20I']

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def clean_records(frame: pd.DataFrame) -> list[dict]:
    return json.loads(frame.to_json(orient='records'))

def safe_rate(numerator, denominator):
    if pd.isna(numerator) or pd.isna(denominator) or denominator == 0:
        return None
    return float(numerator) / float(denominator) * 100.0


In [ ]:
DATA = ROOT / 'SupeRR Selector Season 2 | Data Pack'
competitions = pd.read_csv(DATA / 'competitions.csv')
competition_players = pd.read_csv(DATA / 'competition_team_player_mapping.csv')
batter_formats = pd.read_csv(ROOT / 'phase4_outputs/phase4_batter_format_metrics.csv')
bowler_formats = pd.read_csv(ROOT / 'phase4_outputs/phase4_bowler_format_metrics.csv')
batter_combined = pd.read_csv(ROOT / 'phase4_outputs/phase4_batter_combined_scores.csv')
bowler_combined = pd.read_csv(ROOT / 'phase4_outputs/phase4_bowler_combined_scores.csv')
official_rankings = pd.read_csv(ROOT / 'phase5_outputs/phase5_official_rankings.csv')
bootstrap = pd.read_csv(ROOT / 'phase5_outputs/phase5_bootstrap_summary.csv')
selected_evidence = pd.read_csv(ROOT / 'phase6_outputs/phase6_selected_player_evidence.csv')
selected_formats = pd.read_csv(ROOT / 'phase6_outputs/phase6_selected_format_evidence.csv')
phase7a_bundle = json.loads((ROOT / 'phase7_outputs/phase7_canonical_bundle.json').read_text(encoding='utf-8'))
phase7a_signature = phase7a_bundle['metadata']['official_ranking_signature']

combined = pd.concat([batter_combined, bowler_combined], ignore_index=True)
format_metrics = pd.concat([batter_formats, bowler_formats], ignore_index=True, sort=False)
candidate_players = combined[['player_id', 'player_name']].drop_duplicates().sort_values('player_id').reset_index(drop=True)

t20i_rosters = competition_players.merge(competitions[['comp_id', 'match_format']], on='comp_id', how='left')
t20i_rosters = t20i_rosters.loc[t20i_rosters['match_format'].eq('T20I'), ['player_id', 'player_name', 'team_name']].drop_duplicates()
national_teams = sorted(t20i_rosters['team_name'].dropna().unique().tolist())
len(national_teams), national_teams


In [ ]:
# Reliable candidates without a national-team record in the supplied 2022–2026 T20I mappings.
# These values affect filtering only; they never affect ratings or eligibility.
SUPPLEMENTAL_NATIONAL_TEAMS = {
    8: 'India',
    198: 'India',
    395: 'India',
    781: 'West Indies',
    1071: 'India',
    2411: 'India',
    2756: 'India',
    2878: 'India',
    36982: 'India',
    3782: 'India',
    3926: 'India',
    4219: 'India',
    4246: 'India',
    4265: 'India',
    5039: 'India',
    5577: 'India',
    6161: 'India',
    6460: 'India',
    6669: 'India',
    7059: 'India',
    7327: 'India',
    7446: 'India',
    8530: 'India',
    8746: 'India',
    19714: 'India',
}

direct_mapping = t20i_rosters.loc[t20i_rosters['player_id'].isin(candidate_players['player_id'])].copy()
mapping_counts = direct_mapping.groupby('player_id')['team_name'].nunique()
multi_team_ids = mapping_counts.loc[mapping_counts.gt(1)].index.tolist()
if multi_team_ids:
    raise ValueError(f'Ambiguous national-team mappings: {multi_team_ids}')

direct_mapping = direct_mapping[['player_id', 'team_name']].drop_duplicates('player_id')
national_mapping = candidate_players.merge(direct_mapping, on='player_id', how='left')
national_mapping['mapping_source'] = np.where(national_mapping['team_name'].notna(), 'Supplied T20I competition mapping', None)
missing_before = national_mapping.loc[national_mapping['team_name'].isna(), 'player_id'].astype(int).tolist()
for player_id in missing_before:
    if player_id in SUPPLEMENTAL_NATIONAL_TEAMS:
        mask = national_mapping['player_id'].eq(player_id)
        national_mapping.loc[mask, 'team_name'] = SUPPLEMENTAL_NATIONAL_TEAMS[player_id]
        national_mapping.loc[mask, 'mapping_source'] = 'Audited national-team supplement'

national_mapping = national_mapping.rename(columns={'team_name': 'national_team'})
national_mapping['phase7b_model_version'] = MODEL_VERSION
national_mapping.to_csv(OUT / 'national_team_mapping.csv', index=False)
national_mapping['mapping_source'].value_counts(dropna=False)


In [ ]:
selected_lookup = selected_evidence.set_index(['role', 'player_id']).to_dict(orient='index')
official_lookup = official_rankings.set_index(['role', 'player_id']).to_dict(orient='index')
team_lookup = national_mapping.set_index('player_id')['national_team'].to_dict()
mapping_source_lookup = national_mapping.set_index('player_id')['mapping_source'].to_dict()

bootstrap_lookup = {}
for row in bootstrap.itertuples(index=False):
    bootstrap_lookup.setdefault((row.role, int(row.player_id)), {})[row.bootstrap_mode] = {
        'rankedFrequency': float(row.ranked_frequency),
        'ratingMedian': None if pd.isna(row.rating_median) else float(row.rating_median),
        'ratingLow': None if pd.isna(row.rating_p025) else float(row.rating_p025),
        'ratingHigh': None if pd.isna(row.rating_p975) else float(row.rating_p975),
        'rankMedian': None if pd.isna(row.rank_median) else float(row.rank_median),
        'rankLow': None if pd.isna(row.rank_p025) else float(row.rank_p025),
        'rankHigh': None if pd.isna(row.rank_p975) else float(row.rank_p975),
        'top10Frequency': float(row.top10_frequency),
    }

BATTER_WEIGHTS = {
    'scoringQuality': ('scoring_quality_z', 0.45),
    'survivalQuality': ('survival_quality_z', 0.25),
    'scoringPressureGradient': ('scoring_pressure_gradient_z', 0.20),
    'survivalPressureGradient': ('survival_pressure_gradient_z', 0.10),
}
BOWLER_WEIGHTS = {
    'wicketQuality': ('wicket_quality_z', 0.45),
    'runSuppressionQuality': ('run_suppression_quality_z', 0.30),
    'wicketPressureGradient': ('wicket_pressure_gradient_z', 0.15),
    'runSuppressionPressureGradient': ('run_suppression_pressure_gradient_z', 0.10),
}

def format_payload(metric_row: pd.Series, role: str) -> dict:
    weights = BATTER_WEIGHTS if role == 'Batter' else BOWLER_WEIGHTS
    contributions = {}
    for label, (column, weight) in weights.items():
        value = metric_row.get(column)
        contributions[label] = None if pd.isna(value) else float(value) * weight

    if role == 'Batter':
        comparisons = {
            'primaryLabel': 'Runs per 100 records',
            'primaryHigh': safe_rate(metric_row.get('high_pressure_runs'), metric_row.get('high_records')),
            'primaryOrdinary': safe_rate(metric_row.get('ordinary_runs'), metric_row.get('ordinary_records')),
            'secondaryLabel': 'Survival rate',
            'secondaryHigh': None if safe_rate(metric_row.get('high_dismissals'), metric_row.get('high_records')) is None else 100 - safe_rate(metric_row.get('high_dismissals'), metric_row.get('high_records')),
            'secondaryOrdinary': None if safe_rate(metric_row.get('ordinary_dismissals'), metric_row.get('ordinary_records')) is None else 100 - safe_rate(metric_row.get('ordinary_dismissals'), metric_row.get('ordinary_records')),
            'highQualityResidual': None if pd.isna(metric_row.get('high_scoring_residual_raw_per100')) else float(metric_row.get('high_scoring_residual_raw_per100')),
            'pressureGradient': None if pd.isna(metric_row.get('scoring_pressure_gradient_raw_per100')) else float(metric_row.get('scoring_pressure_gradient_raw_per100')),
        }
    else:
        comparisons = {
            'primaryLabel': 'Wickets per 100 records',
            'primaryHigh': safe_rate(metric_row.get('high_pressure_wickets'), metric_row.get('high_records')),
            'primaryOrdinary': safe_rate(metric_row.get('ordinary_wickets'), metric_row.get('ordinary_records')),
            'secondaryLabel': 'Batter runs allowed per 100 clean records',
            'secondaryHigh': safe_rate(metric_row.get('high_batter_runs_allowed'), metric_row.get('high_clean_records')),
            'secondaryOrdinary': safe_rate(metric_row.get('ordinary_batter_runs_allowed'), metric_row.get('ordinary_clean_records')),
            'highQualityResidual': None if pd.isna(metric_row.get('high_wicket_residual_raw_per100')) else float(metric_row.get('high_wicket_residual_raw_per100')),
            'pressureGradient': None if pd.isna(metric_row.get('wicket_pressure_gradient_raw_per100')) else float(metric_row.get('wicket_pressure_gradient_raw_per100')),
        }

    return {
        'format': metric_row['format_group'],
        'eligibility': metric_row['eligibility'],
        'rating': None if pd.isna(metric_row.get('format_rating')) else float(metric_row.get('format_rating')),
        'highRecords': int(metric_row.get('high_records', 0)),
        'ordinaryRecords': int(metric_row.get('ordinary_records', 0)),
        'highMatches': int(metric_row.get('high_matches', 0)),
        'allMatches': int(metric_row.get('all_matches', 0)),
        'contributions': contributions,
        'comparisons': comparisons,
    }


In [ ]:
players = []
for combined_row in combined.sort_values(['role', 'player_id']).itertuples(index=False):
    key = (combined_row.role, int(combined_row.player_id))
    metric_rows = format_metrics.loc[(format_metrics['role'].eq(combined_row.role)) & (format_metrics['player_id'].eq(combined_row.player_id))]
    formats = {row['format_group']: format_payload(row, combined_row.role) for _, row in metric_rows.iterrows()}
    official_info = official_lookup.get(key, {})
    narrative = selected_lookup.get(key)
    players.append({
        'playerId': int(combined_row.player_id),
        'playerName': combined_row.player_name,
        'role': combined_row.role,
        'nationalTeam': team_lookup.get(int(combined_row.player_id)),
        'nationalTeamSource': mapping_source_lookup.get(int(combined_row.player_id)),
        'qualifyingFormats': combined_row.qualifying_formats.split('+'),
        'combinedRating': float(combined_row.combined_rating),
        'globalCombinedRank': int(official_info.get('rank')) if official_info else None,
        'officialTop10': bool(official_info.get('official_top10', False)),
        'formats': formats,
        'uncertainty': bootstrap_lookup.get(key, {}),
        'robustnessClass': None if narrative is None else narrative.get('robustness_class'),
        'narrative': None if narrative is None else {
            'selection': narrative.get('selection_reason'),
            'format': narrative.get('format_interpretation'),
            'pressure': narrative.get('pressure_interpretation'),
            'uncertainty': narrative.get('uncertainty_interpretation'),
        },
    })

players = sorted(players, key=lambda player: (ROLE_ORDER[player['role']], player['globalCombinedRank'] or 9999, player['playerId']))

def rank_players(role: str, modes: list[str], teams: list[str]) -> list[dict]:
    if not modes or any(mode not in MODE_ORDER for mode in modes):
        raise ValueError('At least one valid mode is required')
    use_combined = set(modes) == set(MODE_ORDER)
    ranked = []
    for player in players:
        if player['role'] != role or player['nationalTeam'] not in teams:
            continue
        if use_combined:
            score = player['combinedRating']
        else:
            mode = modes[0]
            evidence = player['formats'].get(mode)
            if not evidence or evidence['eligibility'] != 'Reliable' or evidence['rating'] is None:
                continue
            score = evidence['rating']
        ranked.append({'playerId': player['playerId'], 'playerName': player['playerName'], 'nationalTeam': player['nationalTeam'], 'score': score})
    ranked.sort(key=lambda row: (-row['score'], row['playerId']))
    for index, row in enumerate(ranked, start=1):
        row['rank'] = index
    return ranked

snapshots = []
mode_sets = [['IPL'], ['T20I'], ['IPL', 'T20I']]
for role in ROLE_ORDER:
    for modes in mode_sets:
        team_sets = [('ALL', national_teams)] + [(team, [team]) for team in national_teams]
        for team_label, teams in team_sets:
            ranked = rank_players(role, modes, teams)
            top_ids = [row['playerId'] for row in ranked[:10]]
            snapshots.append({
                'role': role,
                'modes': '+'.join(modes),
                'team_selection': team_label,
                'eligible_players': len(ranked),
                'displayed_players': min(10, len(ranked)),
                'top_player_ids': ','.join(map(str, top_ids)),
                'top10_signature': hashlib.sha256(','.join(map(str, top_ids)).encode('utf-8')).hexdigest(),
            })
snapshot_frame = pd.DataFrame(snapshots)
snapshot_frame['phase7b_model_version'] = MODEL_VERSION
snapshot_frame.to_csv(OUT / 'dashboard_filter_snapshots.csv', index=False)
len(players), len(snapshot_frame)


In [ ]:
SOURCE_PATHS = [
    'phase7_outputs/phase7_canonical_bundle.json',
    'phase4_outputs/phase4_batter_format_metrics.csv',
    'phase4_outputs/phase4_bowler_format_metrics.csv',
    'phase4_outputs/phase4_batter_combined_scores.csv',
    'phase4_outputs/phase4_bowler_combined_scores.csv',
    'phase5_outputs/phase5_official_rankings.csv',
    'phase5_outputs/phase5_bootstrap_summary.csv',
    'phase6_outputs/phase6_selected_player_evidence.csv',
    'phase6_outputs/phase6_selected_format_evidence.csv',
    'SupeRR Selector Season 2 | Data Pack/competitions.csv',
    'SupeRR Selector Season 2 | Data Pack/competition_team_player_mapping.csv',
]
source_hashes = {path: sha256_file(ROOT / path) for path in SOURCE_PATHS}

dashboard_bundle = {
    'metadata': {
        'modelVersion': MODEL_VERSION,
        'period': '2022-2026',
        'modes': MODE_ORDER,
        'defaultModes': MODE_ORDER,
        'nationalTeams': national_teams,
        'defaultNationalTeams': national_teams,
        'roles': ['Batter', 'Bowler'],
        'defaultRole': 'Batter',
        'eligibilityRule': 'Reliable evidence only',
        'bothModesRule': 'Use the validated combined rating when IPL and T20I are both selected',
        'singleModeRule': 'Use the selected format rating only when that format is Reliable',
        'teamRule': 'National-team selection filters the candidate population and never changes a score',
        'rankingRule': 'Unrounded score descending; player ID ascending tie-break',
        'pressureDefinition': 'Pre-delivery structural pressure from wicket strain, recent wicket shock, and stage urgency',
        'officialRankingSignature': phase7a_signature,
        'sourceHashes': source_hashes,
    },
    'players': players,
}
dashboard_json = json.dumps(dashboard_bundle, ensure_ascii=False, separators=(',', ':'))
(OUT / 'dashboard_data.json').write_text(dashboard_json, encoding='utf-8')
(PUBLIC / 'dashboard_data.json').write_text(dashboard_json, encoding='utf-8')
len(dashboard_json)


In [ ]:
validation_rows = []
def gate(name: str, passed: bool, observed, rule: str):
    validation_rows.append({'check_name': name, 'passed': bool(passed), 'observed': observed, 'acceptance_rule': rule, 'status': 'PASS' if passed else 'FAIL', 'phase7b_model_version': MODEL_VERSION})

unresolved = national_mapping.loc[national_mapping['national_team'].isna(), ['player_id', 'player_name']]
supplement_ids = set(national_mapping.loc[national_mapping['mapping_source'].eq('Audited national-team supplement'), 'player_id'].astype(int))
default_batters = rank_players('Batter', MODE_ORDER, national_teams)[:10]
default_bowlers = rank_players('Bowler', MODE_ORDER, national_teams)[:10]
official_batter_ids = official_rankings.loc[(official_rankings['role'].eq('Batter')) & (official_rankings['official_top10'].eq(True))].sort_values('rank')['player_id'].astype(int).tolist()
official_bowler_ids = official_rankings.loc[(official_rankings['role'].eq('Bowler')) & (official_rankings['official_top10'].eq(True))].sort_values('rank')['player_id'].astype(int).tolist()

gate('mode_options_exact', MODE_ORDER == ['IPL', 'T20I'], MODE_ORDER, 'Exactly IPL and T20I; no third Both option')
gate('default_modes_both_selected', set(dashboard_bundle['metadata']['defaultModes']) == {'IPL', 'T20I'}, dashboard_bundle['metadata']['defaultModes'], 'Both mode options selected by default')
gate('all_national_team_options_present', len(national_teams) == 23, len(national_teams), 'All 23 supplied T20I national teams available')
gate('candidate_role_rows', len(players) == 185, len(players), '80 batter and 105 bowler candidate rows')
gate('candidate_unique_players', len(national_mapping) == 165, len(national_mapping), '165 unique reliable candidate players mapped')
gate('national_team_mapping_complete', unresolved.empty, clean_records(unresolved), 'Every reliable candidate has one national team')
gate('national_team_mapping_unambiguous', not multi_team_ids, multi_team_ids, 'No reliable candidate maps to multiple national teams')
gate('supplement_exact_coverage', supplement_ids == set(SUPPLEMENTAL_NATIONAL_TEAMS), {'used': len(supplement_ids), 'declared': len(SUPPLEMENTAL_NATIONAL_TEAMS)}, 'All and only the 25 missing candidates use the audited supplement')
gate('default_batter_top10_matches_official', [row['playerId'] for row in default_batters] == official_batter_ids, [row['playerId'] for row in default_batters], 'Both modes plus all teams reproduces official batter Top 10')
gate('default_bowler_top10_matches_official', [row['playerId'] for row in default_bowlers] == official_bowler_ids, [row['playerId'] for row in default_bowlers], 'Both modes plus all teams reproduces official bowler Top 10')
gate('single_mode_reliable_only', all(player['formats'][mode]['eligibility'] == 'Reliable' for mode in MODE_ORDER for role in ROLE_ORDER for player in players if player['role'] == role and any(row['playerId'] == player['playerId'] for row in rank_players(role, [mode], national_teams))), True, 'Every single-mode ranked player has Reliable evidence for that mode')
gate('both_mode_score_is_combined', all(np.isclose(row['score'], next(player['combinedRating'] for player in players if player['role'] == role and player['playerId'] == row['playerId'])) for role in ROLE_ORDER for row in rank_players(role, MODE_ORDER, national_teams)), True, 'Both selected modes always use combined rating')
gate('multi_team_union_semantics', {row['playerId'] for row in rank_players('Batter', ['IPL'], ['Australia', 'India'])} == ({row['playerId'] for row in rank_players('Batter', ['IPL'], ['Australia'])} | {row['playerId'] for row in rank_players('Batter', ['IPL'], ['India'])}), True, 'Selecting multiple national teams returns their union')
gate('snapshot_coverage', len(snapshot_frame) == 144, len(snapshot_frame), '2 roles x 3 mode states x 24 team states')
gate('official_signature_preserved', dashboard_bundle['metadata']['officialRankingSignature'] == phase7a_signature, dashboard_bundle['metadata']['officialRankingSignature'], 'Phase 7A official ranking signature preserved')
gate('dashboard_json_matches_public_copy', sha256_file(OUT / 'dashboard_data.json') == sha256_file(PUBLIC / 'dashboard_data.json'), sha256_file(OUT / 'dashboard_data.json'), 'Canonical and public dashboard JSON files are byte-identical')

validation = pd.DataFrame(validation_rows)
validation.to_csv(OUT / 'phase7b_validation_gates.csv', index=False)

generated = {
    'dashboard_data.json': ('Static dashboard data contract', len(players)),
    'national_team_mapping.csv': ('Audited candidate national-team mapping', len(national_mapping)),
    'dashboard_filter_snapshots.csv': ('Validated role, mode, and single-team ranking snapshots', len(snapshot_frame)),
    'phase7b_validation_gates.csv': ('Phase 7B validation results', len(validation)),
}
manifest_rows = []
for filename, (purpose, rows) in generated.items():
    path = OUT / filename
    manifest_rows.append({'output_file': path.relative_to(ROOT).as_posix(), 'sha256': sha256_file(path), 'rows_or_items': rows, 'purpose': purpose, 'phase7b_model_version': MODEL_VERSION})
manifest = pd.DataFrame(manifest_rows).sort_values('output_file').reset_index(drop=True)
manifest.to_csv(OUT / 'phase7b_output_manifest.csv', index=False)

failures = validation.loc[validation['status'].eq('FAIL'), 'check_name'].tolist()
if failures:
    raise AssertionError(f'Phase 7B validation failed: {failures}')
print(f'Phase 7B complete: {len(validation)} validation gates passed.')
print(f'Players: {len(players)} role-player rows; national teams: {len(national_teams)}.')
validation[['check_name', 'status', 'observed']]
